# Snowflake x GenAI: LLM Functions: Lesson 4: A Machine Learning Model Based on an LLM

The Document AI setup, and AI_EXTRACT as its replacement.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `SXGA2_LLM_ADMIN` (ACCOUNTADMIN), `SXGA2_LLM_ROLE` (SYSADMIN) and `SXGA2_LLM_WH` (COMPUTE_WH).
- Course code that no longer works runs **as written** and reports its error; a **Substitute** cell with current functions follows it.
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- 🛠️ **Course update needed** marks what has to change in the course itself.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: Gathering Data for the SUMMARIZE() Function

> 🛠️ **Course update needed:** **Document AI is decommissioned** (BCR-2156, 2026-03-16). `CREATE SNOWFLAKE.ML.DOCUMENT_INTELLIGENCE` no longer parses in this account, there is no Document AI page in Snowsight, and `<model>!PREDICT` is gone. The replacement is `AI_EXTRACT`: no model is built or trained, and the questions are passed in `responseFormat`. Lessons 4-6 are rebuilt below around AI_EXTRACT on the same lyric PDFs, with the same seven questions.

> **Course:** The Doc AI infrastructure: a role, a warehouse, a database and a schema, each created with the scripts below, in order.

> ⚠️ **Audit note:** These are account-level steps, so they run as the real ACCOUNTADMIN. Nothing in the AI_EXTRACT flow needs them (AI functions are available to PUBLIC); they are kept because the later course steps and screenshots refer to DOC_AI_ROLE, DOC_AI_WH and DOC_AI_DB.

In [ ]:
%%sql -r create_doc_ai_role
use role accountadmin;
create role doc_ai_role;

grant database role SNOWFLAKE.DOCUMENT_INTELLIGENCE_CREATOR to role doc_ai_role;

set badge_user = current_user();
grant role doc_ai_role to user identifier($badge_user);

In [ ]:
%%sql -r confirm_doc_ai_role
-- LLMM_116, 117: confirm the role and that it's assigned to you
show roles like 'DOC_AI_ROLE';
show grants to user identifier($badge_user);

In [ ]:
%%sql -r create_doc_ai_wh
use role accountadmin;
create warehouse doc_ai_wh with warehouse_size = 'x-small';

grant USAGE, OPERATE, MODIFY on WAREHOUSE doc_ai_wh to role doc_ai_role;

In [ ]:
%%sql -r confirm_doc_ai_wh
-- LLMM_119, 120
show warehouses like 'DOC_AI_WH';
show grants on warehouse doc_ai_wh;

In [ ]:
%%sql -r create_doc_ai_db
use role accountadmin;
create database doc_ai_db;
grant CREATE SCHEMA, MODIFY, USAGE on DATABASE doc_ai_db to role doc_ai_role;

In [ ]:
%%sql -r confirm_doc_ai_db
-- LLMM_122
show databases like 'DOC_AI_DB';
show grants on database doc_ai_db;

> **Course:** 🧪 LAB: Switch Your Role to DOC_AI_ROLE Before Moving Forward! Then create the DOC_AI_SCHEMA and grant privileges (the role must own the schema).

In [ ]:
%%sql -r create_doc_ai_schema
use role doc_ai_role;
use warehouse doc_ai_wh;
create schema doc_ai_db.doc_ai_schema;

grant USAGE on SCHEMA doc_ai_db.doc_ai_schema to role doc_ai_role;

grant CREATE STAGE ON SCHEMA doc_ai_db.doc_ai_schema to role doc_ai_role;

> ⚠️ **Audit note:** The course's last grant (`CREATE SNOWFLAKE.ML.DOCUMENT_INTELLIGENCE on SCHEMA`) is run on its own below; it reports its result because that privilege belongs to the retired feature.

In [ ]:
%%sql -r grant_create_document_intelligence
execute immediate 'begin
  execute immediate ''grant CREATE SNOWFLAKE.ML.DOCUMENT_INTELLIGENCE on SCHEMA doc_ai_db.doc_ai_schema to role doc_ai_role'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

## Page: Your First Machine Learning Model?

> **Course:** 🥋 LAB: Navigate to the Document AI Homepage: AI & ML → Document AI, with DOC_AI_ROLE and DOC_AI_WH, then build a new model (lyrics_1.png, lyrics_2.png).

> 🛠️ **Course update needed:** The Document AI homepage and **Build** button no longer exist. There is no model to build. Retire both images; the section title is also wrong ("A Machine Learning Model Based on an LLM").

## Page: Loading a PDF Lyric Sheet

> **Course:** 🧪 LAB: Upload a Document for Training (Mary Had A Little Lamb by Storybooksong.pdf; LLMM_136-139).

In [ ]:
%%sql -r create_training_stage
-- Substitute for the model-build upload: the PDFs go on a stage instead. The course's stage LYRICS_PDFS is created in Lesson 5, so the training PDFs use their own stage.
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
create stage training_pdfs
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');

In [ ]:
%%sql -r load_training_pdfs
-- Isolation grants (not a course step): the Downloads stand-in stage is in UTIL_DB, which DOC_AI_ROLE cannot otherwise see. Real ACCOUNTADMIN.
use role accountadmin;
grant usage on database UTIL_DB to role doc_ai_role;
grant usage on schema UTIL_DB.PUBLIC to role doc_ai_role;
grant read on stage UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS to role doc_ai_role;
-- SQL equivalent of uploading the six training PDFs from the Downloads folder
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
copy files into @doc_ai_db.doc_ai_schema.training_pdfs
from @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
files = ('Mary Had A Little Lamb by Storybooksong.pdf', 'Beyond the Sea by Bobby Darin.pdf', 'September by Earth, Wind & Fire.pdf', 'Gloria by Laura Branigan.pdf', 'Fortnight by Taylor Swift.pdf', 'Cutie Pie by One Way.pdf');
alter stage training_pdfs refresh;
list @training_pdfs;

## Page: Choosing Values & Asking the LLM Questions

> **Course:** 📓 Define VALUES as future table columns, each with a question: song_title *What is the name of this song?*; performing_artist *Who performed this version of the song?*; lyrics *What are the lyrics to this song?*; authors *Who wrote this song?*; lyrics_copyright_holders *Who holds the copyright to the lyrics?*; lyric_licensor_provider *Who licensed and provided the lyrics?*; source *Who provided the pdf document?* The LLM gives an answer and a confidence score; approving it sets the score to 1.00 and "the model will learn from the feedback loop" (LLMM_141-150).

> ⚠️ **Audit note:** `AI_EXTRACT` returns the answers only, no confidence score, and nothing is learned from approvals. Six of the seven questions become `responseFormat` keys.

> ⚠️ **Audit note:** **The `lyrics` question cannot be asked with AI_EXTRACT.** Verified 2026-11-14: for any copyrighted lyric sheet (every PDF except Mary Had A Little Lamb) `AI_EXTRACT` answers "AI_EXTRACT failed: the model refused to answer" and returns no `response` at all, even when `lyrics` is the only question; with the lyrics question included, the other six answers are lost too. `AI_PARSE_DOCUMENT` returns the full text of the same PDFs, so the lyrics value comes from it. The function below stands in for the course's model, `LYRICS_EXTRACTOR`: the six metadata questions go to `AI_EXTRACT`, the lyrics come from `AI_PARSE_DOCUMENT` (OCR mode), and the result is one object shaped like `AI_EXTRACT`'s own, `{error, response:{...7 values}}`. The lyrics value includes the title line and the credits at the foot of the page, because it is the whole page text.

In [ ]:
%%sql -r create_lyrics_extractor
-- Substitute for the course's LYRICS_EXTRACTOR model (called here as a function, not with !PREDICT)
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
create or replace function doc_ai_db.doc_ai_schema.lyrics_extractor(f file) returns object as
$$ (select object_construct_keep_null('error', e:error, 'response', object_insert(e:response, 'lyrics', ai_parse_document(f, {'mode':'OCR'}):content::varchar))
    from (select ai_extract(file => f, responseFormat => {'song_title':'What is the name of this song?','performing_artist':'Who performed this version of the song?','authors':'Who wrote this song?','lyrics_copyright_holders':'Who holds the copyright to the lyrics?','lyric_licensor_provider':'Who licensed and provided the lyrics?','source':'Who provided the pdf document?'}) as e)) $$;

In [ ]:
%%sql -r extract_first_pdf
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
select to_varchar(lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.training_pdfs', 'Mary Had A Little Lamb by Storybooksong.pdf'))) as extracted;

> 🛠️ **Course update needed:** Rewrite these four pages (4.4 to 4.6) as one `responseFormat` lab. Drop the lyrics-field instruction to "delete two answers and edit the other": AI_EXTRACT returns the lyrics as one value.

## Page: Continue Training the Model

> **Course:** 🧪 LAB: Load the Next Lyrics PDF and Use it to Keep Training Your Model (Beyond the Sea by Bobby Darin.pdf). *For us, only the last value was not correct. It was empty.*

In [ ]:
%%sql -r extract_second_pdf
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
select to_varchar(lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.training_pdfs', 'Beyond the Sea by Bobby Darin.pdf'))) as extracted;

## Page: More PDFs for Training

> **Course:** 🧪 LAB: Download 4 More Files (September, Gloria, Fortnight, Cutie Pie). *The Snowflake Documentation suggests 20 training cases, but we'll just do 4 more.*

In [ ]:
%%sql -r extract_four_more
use role doc_ai_role;
use warehouse doc_ai_wh;
use schema doc_ai_db.doc_ai_schema;
select relative_path,
  to_varchar(lyrics_extractor(to_file('@doc_ai_db.doc_ai_schema.training_pdfs', relative_path))) as extracted
from directory(@training_pdfs)
where relative_path in ('September by Earth, Wind & Fire.pdf', 'Gloria by Laura Branigan.pdf', 'Fortnight by Taylor Swift.pdf', 'Cutie Pie by One Way.pdf');

> 🛠️ **Course update needed:** "Suggests 20 training cases" was Document AI model-build guidance and does not apply to AI_EXTRACT.